# IBMF vs numerical integration of the Lotka-Volterra equations
Outputs of `LV_ODE_countFP` (`--method ibmf` and `--method ode`) on the same graph and from the same initial conditions,
compared with the functions of `compare_ODE_IBMF.py`. Figure for Sec. 4 of the notes.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, "/home/david/VSCProjects/Ecology/processing_data/IBMF")
from compare_ODE_IBMF import read_dir, load_graph, match, stability

path_runs = "/run/media/david/Data/Research/Ecology/Results/IBMF/ODE_vs_IBMF"
path_out = "/run/media/david/Data/Research/Ecology/Results/Plots_counting_fixed_points/ode_vs_ibmf"
graph_exe = "/home/david/VSCProjects/Ecology/Debug/print_graph_IBMF.out"
os.makedirs(path_out, exist_ok=True)
# (label, directory prefix, N, mu, sigma, seed of the graph); IBMF runs in <prefix>_ibmf, integration in <prefix>_<ode_dir>
# ode_v2: Cash-Karp, lambda = 1e-6, convergence max_i |d ln n_i/dt| < 1e-6, extinct = negative field
cases = [("0.000", "s0", 1024, 0.370, 0.0, 14), ("0.150", "s15", 1024, 0.390, 0.15, 2)]
ode_dir = "ode_v2"


def set_style():
    plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "stix", "axes.linewidth": 1.2})


def format_axis(ax, tick_fontsize=13):
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_fontsize, length=3)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_fontsize, length=5)

In [ ]:
from matplotlib.ticker import MaxNLocator
# 2 x 3 figure: rows sigma = 0 and 0.15; columns phi_ODE vs phi_IBMF, P(m_bar) over the runs, stability
set_style()
color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
fig, axs = plt.subplots(2, 3, figsize=(15, 8.4), dpi=150)
labels = iter('abcdef')
for row, (slab, pre, N, mu, sigma, g) in enumerate(cases):
    I, O = read_dir(f"{path_runs}/{pre}_ibmf"), read_dir(f"{path_runs}/{pre}_{ode_dir}")
    n_init = I['attempts']
    A = load_graph(graph_exe, N, mu, sigma, 0.0, g)
    head = rf"$\sigma={float(slab):.2f}$, $N={N}$, $\mu={mu:.3f}$"
    # (a, d) phi of the fixed points found by both methods
    ax = axs[row, 0]
    io = match(I['M'], O['M'])
    common = np.nonzero(io >= 0)[0]
    pI, pO = I['counts'][common] / n_init, O['counts'][io[common]] / n_init
    ax.loglog(pI, pO, 'o', color=color_cycle[0], ms=5, label=f"{len(common)} fixed points in common")
    lim = np.array([0.3 / n_init, 1.0])
    ax.plot(lim, lim, color='gray', ls=':', lw=1)
    ax.set_xlabel(r"$\phi_{\rm IBMF}$", fontsize=16); ax.set_ylabel(r"$\phi_{\rm ODE}$", fontsize=16)
    ax.set_title(f"({next(labels)}) " + head, fontsize=13)
    ax.legend(loc='upper left', fontsize=10, frameon=False)
    # (b, e) distribution over the runs of the mean abundance of the fixed point reached
    ax = axs[row, 1]
    # m_bar of each distinct fixed point; an ODE fixed point also found by IBMF takes the IBMF value, so that the
    # histograms compare frequencies (the ODE vectors have m_bar lower by O(lambda), an effect of the immigration)
    mbar_fp = {'ibmf': I['M'].mean(1), 'ode': O['M'].mean(1).copy()}
    oi = match(O['M'], I['M'])
    mbar_fp['ode'][oi >= 0] = mbar_fp['ibmf'][oi[oi >= 0]]
    mb = {}
    for name, D in (('ibmf', I), ('ode', O)):
        ok = D['log'][:, 7] >= 0
        mb[name] = mbar_fp[name][D['log'][ok, 7].astype(int)]
    bins = np.linspace(min(mb['ibmf'].min(), mb['ode'].min()), max(mb['ibmf'].max(), mb['ode'].max()), 41)
    for name, lab, col in (('ibmf', 'IBMF', color_cycle[0]), ('ode', 'ODE', color_cycle[1])):
        ax.hist(mb[name], bins=bins, histtype='step', lw=1.6, density=True, color=col, label=lab)
    ax.set_xlabel(r"$\bar m$", fontsize=16); ax.set_ylabel(r"$P(\bar m)$", fontsize=16)
    ax.set_title(f"({next(labels)}) " + head, fontsize=13)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)
    ax.ticklabel_format(axis='x', useOffset=False)
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.legend(loc='upper left', fontsize=10, frameon=False)
    # (c, f) stability: largest field of an extinct species vs smallest real part of eig(D_S M_SS)
    ax = axs[row, 2]
    for name, D, lab, col, mk in (('ibmf', I, 'IBMF', color_cycle[0], 'o'), ('ode', O, 'ODE', color_cycle[1], 's')):
        s = np.array([stability(A, m, sigma == 0) for m in D['M']])
        ax.plot(s[:, 1], s[:, 0], mk, ms=5, mfc='none', color=col, label=lab)
    ax.axvline(0, color='gray', lw=0.8, ls=':'); ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel(r"$\Lambda_{\min}$", fontsize=16); ax.set_ylabel(r"$f^{\rm ext}_{\max}$", fontsize=16)
    ax.set_title(f"({next(labels)}) " + head, fontsize=13)
    ax.ticklabel_format(style='sci', scilimits=(0, 0), useMathText=True)
    ax.legend(loc='lower right', fontsize=10, frameon=False)
for ax in axs.flat:
    format_axis(ax)
fig.tight_layout(w_pad=1.5)
fig.savefig(f"{path_out}/fig_ode_vs_ibmf.png")
plt.show()

## Ten graphs, window of mu (cluster runs)
Averages over graphs 1-10 of the per-task analysis files: $n_{\rm fixed}/n_{\rm conv}$ at the four checkpoints, fraction of runs reaching a fixed point found by both methods, run-by-run agreement and $\pi_{\rm same}=\sum_k c_k(c_k-1)/[n(n-1)]$. Figure for Sec. 4 of the notes.

In [ ]:
from matplotlib.lines import Line2D
import glob, json
# cluster runs (run_LV_ODE_vs_IBMF.sh): graphs 1-10, N = 1024, 2048 initial conditions, both methods;
# one json + npz per (sigma, mu, graph) written by analyse_ODE_IBMF_task.py
path_cluster = "/run/media/david/Data/Research/Ecology/Results/IBMF/Counting_fixed_points/IBMF_vs_ODE/analysis"
path_notes_fig = "/run/media/david/Data/Research/Ecology/Notes/Counting_fixed_points/Figures"


def load_cluster(path):
    """per (sigma, mu): arrays over graphs of the quantities plotted"""
    out = {}
    for f in sorted(glob.glob(f"{path}/*.json")):
        r = json.load(open(f)); z = np.load(f.replace('.json', '.npz')); n = r['n_init']
        q = {}
        for m in ('ibmf', 'ode'):
            q[f'{m}_ninit'] = z[f'{m}_ninit']
            q[f'{m}_ratio'] = z[f'{m}_nfixed'] / z[f'{m}_nconv']
            c = np.rint(z[f'{m}_phi'] * n)
            q[f'{m}_same'] = (c * (c - 1)).sum() / (n * (n - 1))       # prob. that two runs reach the same fixed point
            q[f'{m}_common'] = r[f'phi_{m}_in_common']
        q['runbyrun'] = r['run_by_run_same']
        q['indep'] = (z['phi_ibmf_common'] * z['phi_ode_common']).sum()
        out.setdefault((r['sigma'], r['mu']), []).append(q)
    return out

def stat(lst, key):
    a = np.array([q[key] for q in lst], float)
    return a.mean(0), a.std(0, ddof=1) / np.sqrt(len(a))



In [ ]:
# 3 x 2 figure: columns sigma = 0 and 0.15; rows n_fixed/n_conv, fractions of runs, pi_same (averages over graphs)
data = load_cluster(path_cluster)
set_style()
cc = plt.rcParams['axes.prop_cycle'].by_key()['color']
fig, axs = plt.subplots(3, 2, figsize=(10, 11.4), dpi=150)
for col, sigma in enumerate((0.0, 0.15)):
    mus = np.array(sorted(mu for (s, mu) in data if s == sigma))
    rows = [data[(sigma, mu)] for mu in mus]
    head = rf"$\sigma={sigma:.2f}$, $N=1024$"
    # (a, b) n_fixed / n_conv for the four checkpoints
    ax = axs[0, col]
    ninit = rows[0][0]['ibmf_ninit']
    for k, ni in enumerate(ninit):
        for m, ls, mfc in (('ibmf', '-', None), ('ode', '--', 'none')):
            y = np.array([stat(r, f'{m}_ratio')[0][k] for r in rows]); e = np.array([stat(r, f'{m}_ratio')[1][k] for r in rows])
            ax.errorbar(mus, y, yerr=e, marker='o' if m == 'ibmf' else 's', ls=ls, lw=1.4, ms=5, mfc=mfc, color=cc[k], capsize=2)
    h1 = [Line2D([], [], color=cc[k], lw=1.4, label=rf"$n_{{\rm init}}={ni}$") for k, ni in enumerate(ninit)]
    h2 = [Line2D([], [], color='k', marker='o', ls='-', ms=5, label='IBMF'),
          Line2D([], [], color='k', marker='s', mfc='none', ls='--', ms=5, label='ODE')]
    l1 = ax.legend(handles=h1, loc='upper left', fontsize=9, frameon=False); ax.add_artist(l1)
    ax.legend(handles=h2, loc='lower right', fontsize=9, frameon=False)
    ax.set_ylabel(r"$n_{\rm fixed}/n_{\rm conv}$", fontsize=16); ax.set_ylim(-0.03, 1.05)
    ax.set_title(f"({'ab'[col]}) " + head, fontsize=14)
    # (c, d) fraction of runs that reach a fixed point found by both methods; run-by-run agreement
    ax = axs[1, col]
    for key, lab, c, mk, mfc, ls in (('ibmf_common', 'IBMF, fixed point found by both methods', cc[0], 'o', None, '-'),
                                     ('ode_common', 'ODE, fixed point found by both methods', cc[1], 's', 'none', '--'),
                                     ('runbyrun', 'same fixed point with both methods', cc[2], 'D', None, '-')):
        y, e = np.array([stat(r, key) for r in rows]).T
        ax.errorbar(mus, y, yerr=e, marker=mk, mfc=mfc, ls=ls, lw=1.4, ms=5, color=c, capsize=2, label=lab)
    y = np.array([stat(r, 'indep')[0] for r in rows])
    ax.plot(mus, y, ls=':', lw=1.6, color=cc[2])
    ax.set_ylabel("fraction of runs", fontsize=16); ax.set_ylim(-0.03, 1.42); ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_title(f"({'cd'[col]}) " + head, fontsize=14)
    ax.legend(loc='upper right', fontsize=9, frameon=False)
    ax.plot([0.04, 0.10], [0.10, 0.10], ls=':', lw=1.6, color=cc[2], transform=ax.transAxes)
    ax.text(0.11, 0.10, r"$\sum\phi_{\rm IBMF}\,\phi_{\rm ODE}$", transform=ax.transAxes, fontsize=11, va='center')
    # (e, f) probability that two runs reach the same fixed point
    ax = axs[2, col]
    for m, lab, c, mk, mfc, ls in (('ibmf', 'IBMF', cc[0], 'o', None, '-'), ('ode', 'ODE', cc[1], 's', 'none', '--')):
        y, e = np.array([stat(r, f'{m}_same') for r in rows]).T
        ax.errorbar(mus, y, yerr=e, marker=mk, mfc=mfc, ls=ls, lw=1.4, ms=5, color=c, capsize=2, label=lab)
    ax.set_yscale('log')
    ax.set_ylabel(r"$\pi_{\rm same}$", fontsize=16)
    ax.set_title(f"({'ef'[col]}) " + head, fontsize=14)
    ax.legend(loc='upper right', fontsize=10, frameon=False)
for ax in axs.flat:
    ax.set_xlabel(r"$\mu$", fontsize=16)
    format_axis(ax)
fig.tight_layout()
fig.savefig(f"{path_out}/fig_ode_vs_ibmf_mu.pdf")
fig.savefig(f"{path_notes_fig}/fig_ode_vs_ibmf_mu.pdf")
plt.show()
